## Training with classic machine learning models

In [5]:
import pandas as pd

snapchat_df = pd.read_json("labeled_snapchat_comparison.jsonl", lines=True)
youtube_df = pd.read_json("labeled_youtube_comparison.jsonl", lines=True)

snapchat_df["source"] = "snapchat"
youtube_df["source"] = "youtube"

df = pd.concat([snapchat_df, youtube_df], ignore_index=True)

# Flatten the body text
df["text"] = df["body"].apply(lambda x: " ".join(x))

# Use LLM for ground truth
df["label"] = df["llm_sentiment"].str.lower().str.strip()

# Map to integers
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label_id"] = df["label"].map(label_map)
print(df['label'].value_counts())
# Drop NaNs or unknowns
df = df.dropna(subset=["label_id", "text"])


print(df.head())

label
negative    1364
neutral      472
positive     166
Name: count, dtype: int64
                                               title  \
0    Should Indonesia ban social media for children?   
1  Should Indonesia ban social media for children...   
2  Social Media's Grip on Teen Lives: A Deep Dive...   
3  Australia: Under-16s banned from social media ...   
4                     Breaking the Doom-Scroll Cycle   

                                                body  \
0  [S, ocial media addiction is becoming an incre...   
1  [S, ocial media addiction is becoming an incre...   
2  [A recent report by the Pew Research Centre hi...   
3  [From November 2025, Australians under the age...   
4  [This article is written by a student writer f...   

                                            provider  \
0  {'name': 'thejakartapost.com', 'domain': 'thej...   
1  {'name': 'thejakartapost.com', 'domain': 'thej...   
2  {'name': 'devdiscourse.com', 'domain': 'devdis...   
3  {'name': 'sxm-ta

## Statistics

In [6]:
# Statistics of df (how many lbaels of 0/1/2)
print(df['label'].value_counts())

label
negative    1364
neutral      472
positive     166
Name: count, dtype: int64


## data cleaning

In [7]:
import string
import re

In [8]:

def cleaning(text):        
    # converting to lowercase, removing URL links, special characters, punctuations...
    text = text.lower() # converting to lowercase
    text = re.sub(r"\b\d+\b", "", text) # removing number 
    text = re.sub('<.*?>+', '', text) # removing special characters, 
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # punctuations
    text = re.sub('\n', '', text)
    text = re.sub('[’“”…]', '', text)   

    return text
    
df["cleantext"] = df['text'].apply(cleaning)

## split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["cleantext"], df["label_id"], test_size=0.2, random_state=42, stratify=df["label_id"]
)


## Logistic Regression with TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=300)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))


              precision    recall  f1-score   support

    negative       0.89      0.95      0.92       273
     neutral       0.80      0.78      0.79        95
    positive       1.00      0.48      0.65        33

    accuracy                           0.87       401
   macro avg       0.90      0.74      0.79       401
weighted avg       0.88      0.87      0.87       401



## Transformer-based models

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Build Dataset object
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})
dataset = Dataset.from_pandas(pd.concat([train_df, test_df]))
dataset = dataset.train_test_split(test_size=0.2)

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = dataset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

# Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()


/home/ys/Documents/584NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-25 23:49:57.782254: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-25 23:49:57.821128: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-25 23:49:59.101708: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on.

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.197500,0.279393,0.907731,0.842149
2,0.035700,0.253254,0.905237,0.833620
3,0.049000,0.253179,0.910224,0.844816


TrainOutput(global_step=603, training_loss=0.28352976235781935, metrics={'train_runtime': 115.019, 'train_samples_per_second': 41.758, 'train_steps_per_second': 5.243, 'total_flos': 631866872673792.0, 'train_loss': 0.28352976235781935, 'epoch': 3.0})

## RoBERTa

In [12]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate


tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
# Build Dataset object
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})
dataset = Dataset.from_pandas(pd.concat([train_df, test_df]))
dataset = dataset.train_test_split(test_size=0.2)

# Tokenize
dataset = dataset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
conf_matrix = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir="./RoBERTa_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()


Map: 100%|██████████| 401/401 [00:00<00:00, 4747.25 examples/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.184500,0.391785,0.912718,0.840374
2,0.094400,0.353652,0.927681,0.863239
3,0.181900,0.340341,0.940150,0.887706
4,0.056100,0.361801,0.940150,0.896332
5,0.002800,0.325960,0.947631,0.898582


TrainOutput(global_step=1005, training_loss=0.08922220992656489, metrics={'train_runtime': 194.7521, 'train_samples_per_second': 41.104, 'train_steps_per_second': 5.16, 'total_flos': 1053111454456320.0, 'train_loss': 0.08922220992656489, 'epoch': 5.0})